# 🏗️ Notebook 1: Notification System — Requirements & Architecture

In this notebook we scope the problem: what does the service do, how much traffic it takes, and the rough shape of the architecture. We keep the code tiny — just a **back-of-the-envelope capacity calculator** — so the ideas sink in.

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

Imagine you work at a company with a mobile app, a website, and an email list.
Many different parts of the product need to **notify users**: a 2FA code at login,
an order-shipped email, a marketing push about a weekend sale.

Instead of each team calling APNs / Twilio / SendGrid themselves, we build **one
central notification service** that everyone calls.

### Functional requirements (what it *does*)

| # | Requirement | Example |
|---|---|---|
| 1 | Other services call `send(user_id, template, channel)` | Payments service asks us to email a receipt |
| 2 | Deliver via **push** (APNs/FCM), **email** (SMTP), **SMS** (Twilio) | Same API, 3 backends |
| 3 | Respect user **preferences** (opt-outs, quiet hours) | Don't SMS at 3 AM |
| 4 | **Retry** on transient provider failures | Twilio 503 → try again in a few seconds |
| 5 | **Idempotent**: duplicate requests don't double-notify | Caller retried after timeout |
| 6 | **Priority**: critical messages jump the queue | 2FA code beats a marketing blast |

### Non-functional requirements (how *well*)

| # | Property | Target |
|---|---|---|
| 1 | Throughput (sustained) | **10,000 notifications/sec** |
| 2 | Throughput (spike, e.g. breaking news) | **100,000/sec** for a few minutes |
| 3 | Latency (high-priority) | p95 < 1s from `send()` to provider |
| 4 | Durability | don't lose messages on process restart |
| 5 | Availability | 99.9% monthly |


In [1]:
# Back-of-the-envelope capacity math.
# This kind of simple calculation is the FIRST thing you do in a design interview.

SUSTAINED_QPS = 10_000            # notifications per second
SPIKE_QPS     = 100_000
AVG_PAYLOAD_B = 500               # bytes per notification (template + vars)
RETENTION_DAYS = 7                # we keep send_log / dedup keys for 7 days

seconds_per_day = 24 * 3600
daily_msgs  = SUSTAINED_QPS * seconds_per_day
storage_gb  = daily_msgs * RETENTION_DAYS * AVG_PAYLOAD_B / (1024**3)

print(f"Daily messages:            {daily_msgs:>15,}")
print(f"7-day storage (send_log):  {storage_gb:>15,.1f} GB")
print(f"Spike fan-out (workers):   roughly {SPIKE_QPS // 1000} workers @ 1k/s each")


Daily messages:                864,000,000
7-day storage (send_log):          2,816.3 GB
Spike fan-out (workers):   roughly 100 workers @ 1k/s each


## High-level architecture

```
   callers (other services)
        │ POST /notify
        ▼
  ┌─────────────┐
  │ API gateway │   ← validates payload, authenticates caller
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐     ┌─────────────┐
  │ Preferences │────▶│ drop if     │   ← opt-outs, quiet hours
  │   check     │     │ opted-out   │
  └──────┬──────┘     └─────────────┘
         │
         ▼
  ┌────────────────────────────┐
  │ Priority queues (Kafka /   │
  │   RabbitMQ / Redis streams)│
  │  ├─ high   (2FA, security) │
  │  ├─ normal (txn receipts)  │
  │  └─ low    (marketing)     │
  └──────┬─────────────────────┘
         │
         ▼
  ┌────────────────────────┐
  │ Dispatcher workers     │   ← per-channel worker pools,
  │ (per-channel pool)     │     retry + DLQ + rate-limit per provider
  └──┬───────┬───────┬─────┘
     │       │       │
     ▼       ▼       ▼
    APNs   SMTP   Twilio
```

### Why this shape?

1. **Queues in the middle** ⇒ callers never block on a slow provider. The API returns
   `202 Accepted` as soon as the message is durably queued.
2. **Separate queues per priority** ⇒ a marketing blast of 10M messages cannot delay
   a 2FA code. Each queue has its own worker pool.
3. **Per-channel worker pools** ⇒ when Twilio is slow, SMS backs up but push and email
   keep flowing.
4. **Preferences checked early** ⇒ we never even queue a message the user opted out of,
   saving downstream work.

We'll build small runnable versions of every one of these pieces in the next notebooks.

## Key trade-offs to remember

| Choice | Pros | Cons |
|---|---|---|
| Async queue vs. sync send | Fast API, survives provider outages | Higher end-to-end latency, harder to tell caller "it failed" |
| One queue per priority | Simple, no starvation | More infra to operate |
| One queue per channel | Channel isolation | Mixing priorities inside the channel queue |
| Dedup in Redis vs. DB | Fast reads | Extra moving part, TTL tuning |

Next up: **Notebook 2** — concrete data model, API contract, and a runnable mini pipeline.